# Phase 1 调试 Notebook — EfficientNetV2-S 2.5D 管线逐项检查

> **用途**: 在修改代码后逐 cell 运行，快速定位问题。每个 cell 独立可执行。

## 检查清单
| # | 检查项 | 预期 |
|---|--------|------|
| 1 | 环境 & 导入 | 无报错 |
| 2 | 配置文件加载 | YAML 解析成功 |
| 3 | DICOM 数据检查 | 切片可读、可排序、可显示 |
| 4 | Dataset 构建 | 样本数 > 0, shape=[5,384,384] |
| 5 | 模型 forward | input [B,5,384,384] → output [B,12] |
| 6 | 损失函数 | 正/负/随机标签 loss 合理 |
| 7 | DataLoader 吞吐 | batch 速度正常 |
| 8 | 过拟合测试 | 单 batch loss 趋近于 0 |
| 9 | 快速干跑 | 2 epochs 不报错，AUC 输出 |

In [ ]:
# ============================================================
# Cell 1: 环境 & 导入
# ============================================================
from __future__ import annotations

import sys
from pathlib import Path

# 确保项目根在 sys.path
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# 检查关键依赖
deps = {}
for name in ["torch", "numpy", "pandas", "yaml", "cv2", "pydicom", "timm", "sklearn"]:
    try:
        __import__(name)
        deps[name] = "✅"
    except ImportError:
        deps[name] = "❌ NOT FOUND"

print(f"Python:  {sys.version}")
print(f"PyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU:   {torch.cuda.get_device_name(0)}")
    print(f"  VRAM:  {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
print()
for k, v in deps.items():
    print(f"  {k:<12s} {v}")

In [ ]:
# ============================================================
# Cell 2: 配置文件加载 & 检查
# ============================================================
import yaml

CONFIG_PATH = PROJECT_ROOT / "configs" / "efficientnet.yaml"
with open(CONFIG_PATH, encoding="utf-8") as f:
    config = yaml.safe_load(f)

# 检查必要 key
required_keys = ["experiment", "paths", "data", "model", "train", "loss", "validation"]
for k in required_keys:
    status = "✅" if k in config else "❌ MISSING"
    print(f"  [{status}] {k}")

print(f"\n  实验名: {config['experiment']['name']}")
print(f"  图像尺寸: {config['data']['image_size']}")
print(f"  切片数:   {config['data']['slice_count']}")
print(f"  平面:     {config['data']['planes']}")
print(f"  Batch:    {config['train']['batch_size']}")
print(f"  Epochs:   {config['train']['epochs']}")
print(f"  LR:       {config['optimizer']['lr']}")
print(f"  Focal γ:  {config['loss']['gamma']}")

# 检查路径是否存在
paths = config["paths"]
print(f"\n  路径检查:")
for k, v in paths.items():
    p = Path(v)
    exists = p.exists()
    icon = "✅" if exists else "⚠️ "
    print(f"    {icon} {k}: {v}")

In [ ]:
# ============================================================
# Cell 3: DICOM 数据检查 — 读取 + 可视化
# ============================================================
from datasets.dicom_loader import read_dicom_series, normalize_dicom

DICOM_ROOT = Path(config["paths"]["dicom_root"])

# 找一个存在的 study/series
studies = sorted(DICOM_ROOT.glob("Study_*"))
print(f"找到 {len(studies)} 个 Study")

# 挑第一个 study 的 Sagittal 系列
for study_dir in studies[:3]:
    for series_dir in sorted(study_dir.iterdir()):
        series_name = series_dir.name
        dcm_count = len(list(series_dir.glob("*.dcm")))
        print(f"  {study_dir.name}/{series_name}: {dcm_count} slices")

# 读取一个 series
sample_study = studies[0]
sample_series = sorted(sample_study.iterdir())[0]  # 第一个 series
plane = "Sagittal" if "Sagittal" in sample_series.name else None

print(f"\n读取: {sample_study.name}/{sample_series.name}")
volume = read_dicom_series(sample_series, plane=plane, image_size=384)
print(f"  Volume shape: {volume.shape}  (N_slices={volume.shape[0]}, H={volume.shape[1]}, W={volume.shape[2]})")
print(f"  Value range:  [{volume.min():.3f}, {volume.max():.3f}]")
print(f"  Mean ± std:   {volume.mean():.3f} ± {volume.std():.3f}")

In [ ]:
# ============================================================
# Cell 3b: 5-slice 堆叠可视化
# ============================================================
fig, axes = plt.subplots(1, 5, figsize=(16, 4))

n_slices = volume.shape[0]
center = n_slices // 2
half = 2  # slice_count//2

for i, offset in enumerate(range(-half, half + 1)):
    idx = max(0, min(n_slices - 1, center + offset))
    axes[i].imshow(volume[idx], cmap="gray")
    axes[i].set_title(f"slice {idx}" + (" (center)" if offset == 0 else ""))
    axes[i].axis("off")

fig.suptitle(f"5-Slice Stack: {sample_study.name}/{sample_series.name}", fontsize=13)
plt.tight_layout()
plt.show()

# 也看一下所有切片的 montage
cols = 8
rows = (n_slices + cols - 1) // cols
fig2, axes2 = plt.subplots(rows, cols, figsize=(16, 2 * rows))
for i in range(rows * cols):
    r, c = i // cols, i % cols
    ax = axes2[r, c] if rows > 1 else axes2[c]
    if i < n_slices:
        ax.imshow(volume[i], cmap="gray")
        ax.set_title(f"{i}", fontsize=8)
    ax.axis("off")
fig2.suptitle(f"All {n_slices} Slices — {sample_series.name}", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 4: Dataset 构建 & 样本检查
# ============================================================
from datasets import Knee25DDataset
from utils import TARGET_COLUMNS

# 加载元数据
series_df = pd.read_csv(config["paths"]["series_csv"])
labels_df = pd.read_csv(config["paths"]["train_csv"])

print(f"Series 元数据: {len(series_df):,} 行")
print(f"Labels:        {len(labels_df):,} 行")
print(f"Series 列:     {list(series_df.columns)}")
print(f"Label 列:      {list(labels_df.columns)}")

# 平面分布
if "Anatomical_Plane" in series_df.columns:
    print(f"\n  平面分布:\n{series_df['Anatomical_Plane'].value_counts().to_string()}")

# 构建 Dataset
ds = Knee25DDataset(
    series_df=series_df,
    labels_df=labels_df,
    dicom_root=config["paths"]["dicom_root"],
    planes=config["data"]["planes"],
    image_size=config["data"]["image_size"],
    slice_count=config["data"]["slice_count"],
    is_train=True,
    fluid_sensitive_only=False,   # 合成数据可能不含此列
    fat_suppression_only=False,
)
print(f"\n  Dataset 样本数: {len(ds):,}")

# 抽样检查
print("\n  --- 前 3 个样本 ---")
for i in range(min(3, len(ds))):
    sample = ds[i]
    img, lbl, uid = sample["image"], sample["labels"], sample["study_uid"]
    pos_count = (lbl > 0).sum().item()
    print(f"  [{i}] study={uid:12s}  image={list(img.shape)}  "
          f"range=[{img.min():.3f}, {img.max():.3f}]  pos_labels={pos_count}/12")

In [ ]:
# ============================================================
# Cell 4b: 标签分布统计
# ============================================================
# 统计整个 dataset 的标签分布
all_labels = []
for i in range(len(ds)):
    all_labels.append(ds[i]["labels"].numpy())
all_labels = np.stack(all_labels)  # [N, 12]

print("标签分布 (切片级):")
print(f"{'Class':<20s} {'Positive':>10s} {'Ratio':>10s}")
print("-" * 42)
for i, col in enumerate(TARGET_COLUMNS):
    pos = all_labels[:, i].sum()
    ratio = pos / len(all_labels)
    bar = "█" * int(ratio * 100)
    print(f"{col:<20s} {int(pos):10d} {ratio:9.2%}  {bar}")

print(f"\n  总样本数: {len(all_labels):,}")
print(f"  每样本平均正类数: {all_labels.sum(axis=1).mean():.2f}")

# Study 级标签分布
study_labels = {}
for i in range(len(ds)):
    sample = ds.samples[i]
    uid = sample["study_uid"]
    if uid not in study_labels:
        study_labels[uid] = sample["labels"]

study_arr = np.stack(list(study_labels.values()))
print(f"\n  Study 数: {len(study_labels)}")
print(f"  有异常的 Study: {(study_arr.sum(axis=1) > 0).sum()} / {len(study_labels)}")

In [ ]:
# ============================================================
# Cell 5: 模型 forward pass 验证
# ============================================================
from models import EfficientNetV2S25D

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# 构建模型
model = EfficientNetV2S25D(
    in_channels=config["model"]["in_channels"],
    num_classes=config["model"]["num_classes"],
    pretrained=True,
    dropout=config["model"]["dropout"],
).to(device)

# 统计参数量
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  总参数量:     {n_params/1e6:.2f}M")
print(f"  可训练参数:   {n_trainable/1e6:.2f}M")

# 测试 forward: random input
print("\n--- Forward pass 测试 ---")
for bs in [1, 2, 8]:
    x = torch.randn(bs, 5, 384, 384).to(device)
    with torch.no_grad():
        out = model(x)
    print(f"  input: {list(x.shape)} → output: {list(out.shape)}")
    assert out.shape == (bs, 12), f"Shape mismatch! Expected ({bs}, 12), got {out.shape}"

print("\n  ✅ Forward pass shapes correct!")

# 测试 extract_features
print("\n--- Feature extraction 测试 ---")
x = torch.randn(4, 5, 384, 384).to(device)
with torch.no_grad():
    feats = model.extract_features(x)
print(f"  input: {list(x.shape)} → features: {list(feats.shape)}")
assert feats.shape == (4, config["model"]["feature_dim"]), \
    f"Feature dim mismatch! Expected (4, {config['model']['feature_dim']}), got {feats.shape}"
print(f"  ✅ Feature extraction OK, dim={feats.shape[1]}")

In [ ]:
# ============================================================
# Cell 5b: 检查 backbone 特征图尺寸
# ============================================================
# 确认 forward_features 输出正确被 GAP 处理
x = torch.randn(2, 5, 384, 384).to(device)

with torch.no_grad():
    # 直接调 backbone
    raw_features = model.backbone.forward_features(x)
    print(f"  Backbone 原始输出:    {list(raw_features.shape)}")
    
    # GAP
    pooled = raw_features.mean(dim=[2, 3])
    print(f"  GAP 后:              {list(pooled.shape)}")
    
    # 完整 forward
    full_out = model(x)
    print(f"  Head 输出 (logits):   {list(full_out.shape)}")

# 检查值域
print(f"\n  Logits range: [{full_out.min():.3f}, {full_out.max():.3f}]")
print(f"  Logits mean ± std: {full_out.mean():.3f} ± {full_out.std():.3f}")
print(f"  After sigmoid range: [{full_out.sigmoid().min():.3f}, {full_out.sigmoid().max():.3f}]")

In [ ]:
# ============================================================
# Cell 6: 损失函数测试
# ============================================================
from losses import FocalBCELoss

criterion = FocalBCELoss(gamma=2.0, alpha=0.25)

# 测试 1: 完美预测
targets_pos = torch.ones(4, 12)       # 全正样本
logits_perfect = torch.full((4, 12), 5.0)  # 高置信度正确
loss1 = criterion(logits_perfect, targets_pos)
print(f"  Perfect prediction (all positive, high confidence): loss={loss1.item():.6f}")

# 测试 2: 完全错误
logits_wrong = torch.full((4, 12), -5.0)  # 高置信度但错误
loss2 = criterion(logits_wrong, targets_pos)
print(f"  Wrong prediction (all positive, conf wrong):      loss={loss2.item():.6f}")

# 测试 3: 随机预测
logits_random = torch.randn(4, 12) * 0.1
loss3 = criterion(logits_random, targets_pos)
print(f"  Random logits (near zero):                         loss={loss3.item():.6f}")

# 测试 4: 全负样本 + 正确预测
targets_neg = torch.zeros(4, 12)
logits_neg_correct = torch.full((4, 12), -5.0)
loss4 = criterion(logits_neg_correct, targets_neg)
print(f"  All negative, correct prediction:                  loss={loss4.item():.6f}")

# 测试 5: 混合标签
targets_mixed = torch.randint(0, 2, (4, 12)).float()
logits_mixed = torch.randn(4, 12)
loss5 = criterion(logits_mixed, targets_mixed)
print(f"  Mixed random labels & logits:                      loss={loss5.item():.6f}")

print("\n  ✅ 损失函数行为正常: 正确→低loss, 错误→高loss, 随机→中等loss")

In [ ]:
# ============================================================
# Cell 7: DataLoader 吞吐测试
# ============================================================
from torch.utils.data import DataLoader
import time

BATCH_SIZE = config["train"]["batch_size"]

loader = DataLoader(
    ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

print(f"DataLoader: batch_size={BATCH_SIZE}, workers=4, pin_memory=True")
print(f"  Total batches: {len(loader)}")

# 测速: 遍历 10 个 batch
n_warmup = 3
n_test = 10

for i, batch in enumerate(loader):
    if i == n_warmup:
        t0 = time.time()
    if i >= n_warmup + n_test:
        break
    imgs = batch["image"]
    lbls = batch["labels"]

elapsed = time.time() - t0
per_batch = elapsed / n_test
imgs_per_sec = BATCH_SIZE / per_batch

print(f"\n  吞吐: {per_batch*1000:.0f} ms/batch, {imgs_per_sec:.0f} samples/sec")
print(f"  VRAM/样本: ~{imgs[0].numel() * 4 / 1024**2:.1f} MB (float32)")

# 检查一个 batch 的内容
batch = next(iter(loader))
print(f"\n  Batch keys:    {list(batch.keys())}")
print(f"  image shape:   {list(batch['image'].shape)}")
print(f"  labels shape:  {list(batch['labels'].shape)}")
print(f"  study_uids:    {batch['study_uid'][:3]}")

In [ ]:
# ============================================================
# Cell 8: 过拟合测试 — 单 batch 训练至 loss→0
# ============================================================
"""
这是最重要的调试检查: 如果模型不能在单 batch 上过拟合,
说明模型、数据或损失函数有问题.
"""

# 取一个小 batch
batch = next(iter(loader))
imgs = batch["image"].to(device)
lbls = batch["labels"].to(device)

print(f"Overfitting test: {imgs.shape[0]} samples, device={device}")

# 新模型 (避免受之前的权重影响)
test_model = EfficientNetV2S25D(
    in_channels=5, num_classes=12, pretrained=True, dropout=0.3,
).to(device)

criterion = FocalBCELoss(gamma=2.0, alpha=0.25)
optimizer = torch.optim.AdamW(test_model.parameters(), lr=1e-3)

# 过拟合训练 (期望 20-50 step 内 loss → 0)
loss_history = []
MAX_STEPS = 150

for step in range(MAX_STEPS):
    test_model.train()
    optimizer.zero_grad()
    
    out = test_model(imgs)
    loss = criterion(out, lbls)
    loss.backward()
    optimizer.step()
    
    loss_history.append(loss.item())
    
    if step % 25 == 0 or loss.item() < 0.001:
        with torch.no_grad():
            from utils import compute_macro_auc
            auc = compute_macro_auc(lbls.cpu().numpy(), out.detach().cpu().numpy())
        print(f"  step {step:4d}: loss={loss.item():.6f}  AUC={auc:.4f}")
    
    if loss.item() < 0.001:
        print(f"  ✅ 过拟合成功! (step {step}, loss < 0.001)")
        break
else:
    print(f"  ⚠️  {MAX_STEPS} steps 未完全过拟合, final loss={loss_history[-1]:.6f}")
    if loss_history[-1] > 0.01:
        print("  ❌ 过拟合测试失败! 检查模型/数据/损失函数")

# 画 loss 曲线
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(loss_history)
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("Overfitting Test — Single Batch")
ax.set_yscale("log")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 9: 快速干跑 — 完整训练 2 epochs
# ============================================================
"""
最终验证: 跑 2 个 epoch 确保训练循环不报错.
不关心 AUC (合成数据 label 随机), 只验证流程.
"""
from train import train_one_epoch, validate_one_epoch
from utils import format_per_class_auc

# 小数据集快速测试
TRAIN_STUDIES = 10
VAL_STUDIES = 4

all_study_ids = sorted(set(ds.samples[s]["study_uid"] for s in range(len(ds))))
train_ids = set(all_study_ids[:TRAIN_STUDIES])
valid_ids = set(all_study_ids[TRAIN_STUDIES:TRAIN_STUDIES + VAL_STUDIES])

# 过滤 series
train_meta = series_df[series_df["StudyInstanceUID"].isin(train_ids)]
valid_meta = series_df[series_df["StudyInstanceUID"].isin(valid_ids)]

print(f"Train studies: {len(train_ids)},  Valid studies: {len(valid_ids)}")

# 构建 dataset
ds_kwargs = dict(
    dicom_root=config["paths"]["dicom_root"],
    planes=config["data"]["planes"],
    image_size=config["data"]["image_size"],
    slice_count=config["data"]["slice_count"],
    fluid_sensitive_only=False,
    fat_suppression_only=False,
)

train_ds = Knee25DDataset(train_meta, labels_df, is_train=True, **ds_kwargs)
valid_ds = Knee25DDataset(valid_meta, labels_df, is_train=False, **ds_kwargs)

print(f"Train samples: {len(train_ds)},  Valid samples: {len(valid_ds)}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

# 模型 & 优化器
model2 = EfficientNetV2S25D(in_channels=5, num_classes=12, pretrained=True).to(device)
criterion = FocalBCELoss(gamma=2.0, alpha=0.25)
optimizer = torch.optim.AdamW(model2.parameters(), lr=2e-4, weight_decay=1e-4)
scaler = torch.amp.GradScaler("cuda") if device == "cuda" else None

# 训练 2 epochs
print(f"\n{'='*55}")
for epoch in range(2):
    t0 = time.time()
    
    train_loss = train_one_epoch(
        model2, train_loader, optimizer, criterion, scaler,
        grad_clip_norm=1.0, device=device,
    )
    val_metrics = validate_one_epoch(model2, valid_loader, criterion, device=device)
    
    elapsed = time.time() - t0
    vram = torch.cuda.max_memory_allocated(device) / 1024**3 if device == "cuda" else 0
    torch.cuda.reset_peak_memory_stats(device)
    
    print(f"Epoch {epoch}: train_loss={train_loss:.4f}  "
          f"val_loss={val_metrics['loss']:.4f}  val_auc={val_metrics['macro_auc']:.4f}  "
          f"⏱ {elapsed:.0f}s  VRAM={vram:.1f}GB")
    print(f"         per-class: {format_per_class_auc(val_metrics['per_class_auc'])}")

print(f"{'='*55}")
print("✅ 2-epoch 干跑完成, 训练循环无报错!")

---
## 调试结论

所有 9 项检查应全部通过。如果有 ❌，对应修改对应模块代码后重新运行该 cell。

### 常见问题速查

| 现象 | 可能原因 | 检查 |
|------|----------|------|
| `forward_features` shape 错误 | 未加 GAP | `features.mean(dim=[2,3])` |
| loss 不下降 | 学习率太低/太高 | 尝试 1e-3 过拟合测试 |
| AUC ≈ 0.5 | 合成数据标签随机 (正常) | 过拟合测试应该 loss→0 |
| CUDA OOM | batch_size 太大 | 减小 batch_size 或 image_size |
| DICOM 读取失败 | pydicom 版本问题 | 加 `force=True` 参数 |
| 切片顺序错乱 | 缺少 ImagePositionPatient | 回退到 InstanceNumber |
| DataLoader 慢 | num_workers=0 | 设为 2-4 |